HieroglyphMatics — Own Lossless Rank-32 CPCR LoRA Submission Notebook

Purpose:

- Create **our own** rank-32 LoRA adapter.
- Use a **lossless Egyptian-hieroglyph byte codec** as dual replay substrate.
- Train on CPCR verified reasoning rows plus mined public rows from `/kaggle/input`.
- Run SFT maximize, then replay low minimum-logprob answer rows.
- Save a real adapter checkpoint and a flat `/kaggle/working/submission.zip`.

This notebook is the Kaggle/cloud training version of the local green smoke path. It builds or loads the verified CPCR + glyph-dual corpus, resolves a locally mounted Nemotron base model from `/kaggle/input`, disables FlashAttention2, installs/uses Omni dependencies when available, trains our own rank-32 LoRA adapter, validates tensors, and writes a flat `/kaggle/working/submission.zip`.



In [ ]:
# =============================================================================
# Cell 1 — Configuration
# =============================================================================
from __future__ import annotations
from pathlib import Path
import os, sys, re, json, csv, ast, math, time, random, shutil, zipfile, hashlib, traceback, subprocess
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from typing import Any, Optional, Iterable

SEED = 918
random.seed(SEED)

IS_KAGGLE = Path('/kaggle').exists()
WORK = Path('/kaggle/working') if IS_KAGGLE else Path.cwd() / 'working'
INPUT = Path('/kaggle/input') if IS_KAGGLE else Path.cwd() / 'input'
BUILD = WORK / 'own_lossless_hieroglyph_rank32_build'
ADAPTER_DIR = WORK / 'own_lossless_hieroglyph_rank32_adapter'
CHECKPOINT_ZIP = WORK / 'own_lossless_hieroglyph_rank32_adapter_checkpoint.zip'
SUBMISSION_ZIP = WORK / 'submission.zip'
MANIFEST_PATH = WORK / 'own_lossless_hieroglyph_rank32_final_manifest.json'
for p in [WORK, BUILD, ADAPTER_DIR]:
    p.mkdir(parents=True, exist_ok=True)

OWN_ADAPTER_ONLY = True
ALLOW_EXISTING_ADAPTER_PACKAGING = False
ALLOW_WARM_START_ADAPTER = False

# Training budget. Set SMOKE_TEST_MODE=True for path/debug only.
SMOKE_TEST_MODE = False
# Keep False for Kaggle/cloud training. Local Linux smoke can set True.
SMOKE_CORPUS_ONLY = False
# Default is offline/local-base-first because Kaggle DNS/HF often fails.
# Set ALLOW_HF_DOWNLOAD=True only if the environment has reliable internet.
ALLOW_HF_DOWNLOAD = False
PREFER_UPLOADED_CORPUS_BUNDLE = True
TRAINING_TIME_BUDGET_SECONDS = 8 * 60 * 60
TRAINING_END_BUFFER_SECONDS = 8 * 60
SAVE_EVERY_SECONDS = 12 * 60

# LoRA / optimizer. Rank must remain <=32 for competition contract.
LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.03
LEARNING_RATE_PHASE1 = 1.5e-4
LEARNING_RATE_PHASE2 = 9.0e-5
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_STEPS_SMOKE = 3
TARGET_MODULES = 'all-linear'

# Replay policy.
SYNTHETIC_BASE_LIMITS = dict(equation=450, numeral=400, unit=400, bit=320, gravity=220, cryptarithm=50)
PUBLIC_SCAN_FILE_LIMIT = 2500
PUBLIC_SCAN_ROW_LIMIT = 250000
MAX_TRAIN_ROWS = 90000
MIN_LOGPROB_EVAL_ROWS = 8192
MIN_LOGPROB_REPLAY_TOP_N = 2400
MIN_LOGPROB_REPLAY_MULTIPLIER = 4
ANCHOR_REPLAY_MIX = 0.35

print(json.dumps({
    'WORK': str(WORK),
    'INPUT': str(INPUT),
    'BUILD': str(BUILD),
    'ADAPTER_DIR': str(ADAPTER_DIR),
    'OWN_ADAPTER_ONLY': OWN_ADAPTER_ONLY,
    'LORA_RANK': LORA_RANK,
    'TRAINING_TIME_BUDGET_SECONDS': TRAINING_TIME_BUDGET_SECONDS,
}, indent=2))




In [ ]:
# =============================================================================
# Cell 2 — Dependency resolver and compatibility guards
# =============================================================================
def run(cmd, check=True, quiet=False):
    print('[RUN]', ' '.join(map(str, cmd)))
    kwargs = {}
    if quiet:
        kwargs.update(stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = subprocess.run(cmd, check=check, **kwargs)
    if quiet and out.stdout:
        print(out.stdout[-4000:])
    return out

def ensure_packages():
    # Import-name -> pip package. Remote Nemotron Omni code may require vision/audio deps.
    deps = [
        ('torch', 'torch'),
        ('safetensors', 'safetensors'),
        ('transformers', 'transformers'),
        ('peft', 'peft'),
        ('accelerate', 'accelerate'),
        ('datasets', 'datasets'),
        ('bitsandbytes', 'bitsandbytes'),
        ('sentencepiece', 'sentencepiece'),
        ('protobuf', 'protobuf'),
        ('open_clip', 'open-clip-torch'),
        ('timm', 'timm'),
        ('torchvision', 'torchvision'),
        ('librosa', 'librosa'),
        ('soundfile', 'soundfile'),
        ('soxr', 'soxr'),
    ]
    missing=[]
    for import_name, pip_name in deps:
        try:
            __import__(import_name)
        except Exception:
            missing.append(pip_name)
    # preserve order and dedupe
    missing=list(dict.fromkeys(missing))
    if missing:
        print('[INFO] missing packages:', missing)
        # Prefer mounted wheelhouses if present, else normal pip. Kaggle internet may be disabled.
        wheel_dirs=[]
        if INPUT.exists():
            for d in INPUT.rglob('*'):
                if d.is_dir() and any(x.suffix == '.whl' for x in d.glob('*.whl')):
                    wheel_dirs.append(d)
        installed=False
        if wheel_dirs:
            for wd in wheel_dirs[:12]:
                try:
                    run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wd)] + missing, check=False)
                    installed=True
                except Exception:
                    pass
        # Last resort. In Kaggle offline sessions this may fail harmlessly; hard import errors will appear later.
        try:
            run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + missing, check=False)
        except Exception as e:
            print('[WARN] dependency install attempt failed:', repr(e))
    else:
        print('[OK] required import stack already present')

ensure_packages()

# Transformers 5.x can reject trusted remote code because of auto-docstring checks.
def disable_transformers_auto_docstring() -> bool:
    try:
        import importlib
        import transformers.utils as hf_utils
        def identity_auto_docstring(obj=None, *args, **kwargs):
            if obj is None:
                return lambda x: x
            return obj
        try:
            mod = importlib.import_module('transformers.utils.auto_docstring')
            if hasattr(mod, 'auto_docstring'):
                mod.auto_docstring = identity_auto_docstring
        except Exception:
            pass
        if hasattr(hf_utils, 'auto_docstring'):
            hf_utils.auto_docstring = identity_auto_docstring
        return True
    except Exception:
        return False

print('[INFO] disabled_auto_docstring=', disable_transformers_auto_docstring())




In [ ]:
# =============================================================================
# Cell 3 — Lossless Egyptian-hieroglyph byte codec
# =============================================================================
# The user-provided hieroglyph polyglot concept is made competition-safe here:
# we do NOT resize tokenizer or save embedding/lm_head. Instead we use Unicode
# Egyptian Hieroglyph characters as a reversible byte-level representation.
# This codec is mathematically lossless: text -> UTF-8 bytes -> glyphs -> text.

HIERO_BYTE_BASE = 0x13000  # Egyptian Hieroglyph block starts at U+13000.
HIERO_BYTE_END = HIERO_BYTE_BASE + 255
HIERO_PREFIX = '𓀀'  # visible sentinel, also U+13000

BYTE_TO_HIERO = {i: chr(HIERO_BYTE_BASE + i) for i in range(256)}
HIERO_TO_BYTE = {chr(HIERO_BYTE_BASE + i): i for i in range(256)}

def text_to_hiero_bytes(text: str) -> str:
    b = text.encode('utf-8')
    return ''.join(BYTE_TO_HIERO[x] for x in b)

def hiero_bytes_to_text(glyphs: str) -> str:
    bs = bytearray()
    for ch in glyphs:
        if ch not in HIERO_TO_BYTE:
            raise ValueError(f'not a codec glyph: U+{ord(ch):04X}')
        bs.append(HIERO_TO_BYTE[ch])
    return bs.decode('utf-8')

def is_codec_hiero(ch: str) -> bool:
    return HIERO_BYTE_BASE <= ord(ch) <= HIERO_BYTE_END

def codec_selftest():
    samples = [
        'What is 7 * 12?',
        'Compute bitwise xor 13 and 7.',
        '𓂀 glyphmatics / CPCR / Nine1Eight',
        'UTF-8: αβγ ꜣ ḥ ḏ 🚀',
    ]
    for s in samples:
        g = text_to_hiero_bytes(s)
        r = hiero_bytes_to_text(g)
        assert r == s, (s, r)
    print('[OK] lossless hieroglyph byte codec selftest passed')

codec_selftest()




In [ ]:
# =============================================================================
# Cell 4 — CPCR deterministic solvers and verified row schema
# =============================================================================
@dataclass
class SolverResult:
    category: str
    answer: str
    verified: bool
    confidence: float
    source: str
    meta: dict

@dataclass
class TrainRow:
    prompt: str
    output: str
    answer: str
    category: str
    cpcr_stage: int
    confidence: float
    source: str
    verified: bool = True
    train_allowed: bool = True
    weight: float = 1.0
    replay_kind: str = 'direct'
    meta: dict = None

CPCR_STAGE = {
    'numeral_system': 0,
    'unit_conversion': 0,
    'physics_gravity': 0,
    'equation_numeric': 1,
    'cryptarithm_deduce': 2,
    'bit_manipulation': 3,
}
CATEGORY_WEIGHTS = {
    'numeral_system': 4.0,
    'unit_conversion': 4.0,
    'physics_gravity': 3.5,
    'equation_numeric': 3.0,
    'cryptarithm_deduce': 3.5,
    'bit_manipulation': 2.8,
}

ALLOWED_AST = {
    ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
    ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod, ast.Pow,
    ast.USub, ast.UAdd, ast.LShift, ast.RShift, ast.BitAnd, ast.BitOr, ast.BitXor,
    ast.Load, ast.Tuple, ast.List,
}

def _eval_ast(node):
    if type(node) not in ALLOWED_AST:
        raise ValueError(f'disallowed AST: {type(node).__name__}')
    if isinstance(node, ast.Expression): return _eval_ast(node.body)
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)): return node.value
        raise ValueError('bad constant')
    if isinstance(node, ast.UnaryOp):
        v=_eval_ast(node.operand)
        if isinstance(node.op, ast.USub): return -v
        if isinstance(node.op, ast.UAdd): return +v
    if isinstance(node, ast.BinOp):
        a,b=_eval_ast(node.left),_eval_ast(node.right)
        if isinstance(node.op, ast.Add): return a+b
        if isinstance(node.op, ast.Sub): return a-b
        if isinstance(node.op, ast.Mult): return a*b
        if isinstance(node.op, ast.Div): return a/b
        if isinstance(node.op, ast.FloorDiv): return a//b
        if isinstance(node.op, ast.Mod): return a%b
        if isinstance(node.op, ast.Pow): return a**b
        if isinstance(node.op, ast.LShift): return int(a)<<int(b)
        if isinstance(node.op, ast.RShift): return int(a)>>int(b)
        if isinstance(node.op, ast.BitAnd): return int(a)&int(b)
        if isinstance(node.op, ast.BitOr): return int(a)|int(b)
        if isinstance(node.op, ast.BitXor): return int(a)^int(b)
    raise ValueError('unsupported expression')

def safe_eval(expr: str):
    # Important: do NOT replace ^ with **. In CPCR bit context, ^ means XOR.
    return _eval_ast(ast.parse(expr, mode='eval'))

def solve_numeric_expression(prompt: str) -> Optional[SolverResult]:
    pl = prompt.lower()
    if any(tok in pl for tok in ['bitwise', ' xor ', 'left shift', 'right shift', '<<', '>>', '&', '|', ' ^ ']):
        return None
    pats=[r'(?:what is|compute|calculate)\s+([0-9\s+\-*/().%]+)\??', r'([0-9]+\s*[+\-*/%]\s*[0-9][0-9\s+\-*/().%]*)']
    for pat in pats:
        m=re.search(pat, pl)
        if not m: continue
        expr=m.group(1).strip().rstrip('? .')
        if not re.fullmatch(r'[0-9\s+\-*/().%]+', expr):
            continue
        try:
            val=safe_eval(expr)
            if isinstance(val, float) and val.is_integer(): val=int(val)
            return SolverResult('equation_numeric', str(val), True, .95, 'safe_ast_arithmetic', {'expr':expr})
        except Exception:
            pass
    return None

def solve_numeral_system(prompt: str) -> Optional[SolverResult]:
    p=prompt.lower()
    m=re.search(r'convert\s+decimal\s+([0-9]+)\s+to\s+binary', p)
    if m: return SolverResult('numeral_system', bin(int(m.group(1)))[2:], True, 1.0, 'deterministic_base_conversion', {})
    m=re.search(r'convert\s+binary\s+([01]+)\s+to\s+decimal', p)
    if m: return SolverResult('numeral_system', str(int(m.group(1),2)), True, 1.0, 'deterministic_base_conversion', {})
    m=re.search(r'what\s+is\s+([01]+)\s+in\s+decimal', p)
    if m: return SolverResult('numeral_system', str(int(m.group(1),2)), True, 1.0, 'deterministic_base_conversion', {})
    return None

def solve_unit_conversion(prompt: str) -> Optional[SolverResult]:
    p=prompt.lower().replace('meters','meter').replace('kilometers','km').replace('hours','hour').replace('minutes','minute')
    conversions=[
        (r'convert\s+([0-9]+(?:\.\d+)?)\s*km\s+to\s+meter', 1000),
        (r'convert\s+([0-9]+(?:\.\d+)?)\s*hour\s+to\s+minute', 60),
        (r'convert\s+([0-9]+(?:\.\d+)?)\s*meter\s+to\s+cm', 100),
        (r'convert\s+([0-9]+(?:\.\d+)?)\s*kg\s+to\s+g', 1000),
    ]
    for pat,mul in conversions:
        m=re.search(pat,p)
        if m:
            val=float(m.group(1))*mul
            ans=str(int(val)) if abs(val-round(val))<1e-9 else str(val)
            return SolverResult('unit_conversion', ans, True, 1.0, 'deterministic_unit_conversion', {'mul':mul})
    return None

def solve_gravity(prompt: str) -> Optional[SolverResult]:
    p=prompt.lower()
    if 'weight' not in p or ('newton' not in p and ' n' not in p): return None
    m=re.search(r'mass\s+is\s+([0-9]+(?:\.\d+)?)\s*kg', p) or re.search(r'mass\s*[:=]\s*([0-9]+(?:\.\d+)?)', p)
    if not m: return None
    mass=float(m.group(1)); g=9.8
    ans=mass*g
    # Competition tolerance handles numeric; use compact no trailing .0 when possible.
    out=str(int(ans)) if abs(ans-round(ans))<1e-9 else f'{ans:.10g}'
    return SolverResult('physics_gravity', out, True, .98, 'earth_weight_g_9_8', {'mass':mass})

def solve_bit_manipulation(prompt: str) -> Optional[SolverResult]:
    p=prompt.lower()
    bit_signal=any(tok in p for tok in ['bitwise',' xor ','left shift','right shift','<<','>>','&','|',' ^ '])
    if not bit_signal: return None
    expr=None
    m=re.search(r'(?:bitwise\s+)?(xor|and|or)\s+([0-9]+)\s+(?:and|with)\s+([0-9]+)', p)
    if m:
        op={'xor':'^','and':'&','or':'|'}[m.group(1)]
        expr=f'{m.group(2)} {op} {m.group(3)}'
    if expr is None:
        m=re.search(r'(?:left shift|right shift)\s+([0-9]+)\s+(?:by\s+)?([0-9]+)', p)
        if m: expr=f"{m.group(1)} {'<<' if 'left shift' in p else '>>'} {m.group(2)}"
    if expr is None:
        m=re.search(r'(?:what is|compute|calculate)\s+([0-9\s+\-*/()%^<>&|~]+)', p)
        if m: expr=m.group(1).strip().rstrip('? .')
    if not expr or not re.search(r'(<<|>>|[&|^~])', expr): return None
    try:
        val=safe_eval(expr)
        return SolverResult('bit_manipulation', str(int(val)), True, .99, 'safe_ast_bitwise', {'expr':expr})
    except Exception:
        return None

def solve_cryptarithm(prompt: str) -> Optional[SolverResult]:
    p=prompt.lower()
    # Conservative verified anchor used only for known symbolic pattern.
    if 'cryptarithm' in p and re.search(r'a\s*\+\s*a\s*=\s*b', p):
        return SolverResult('cryptarithm_deduce', '2', True, .96, 'verified_symbolic_anchor', {'rule':'A+A=B; A=1=>B=2'})
    return None

def solve_any(prompt: str) -> Optional[SolverResult]:
    # CPCR: verified/easy first, risky bit later.
    for fn in [solve_numeral_system, solve_unit_conversion, solve_gravity, solve_numeric_expression, solve_cryptarithm, solve_bit_manipulation]:
        r=fn(prompt)
        if r is not None: return r
    return None

def make_row(prompt: str, r: SolverResult, source='residual_verified_solver', replay_kind='direct', weight=None) -> TrainRow:
    weight = CATEGORY_WEIGHTS.get(r.category, 1.0) if weight is None else weight
    return TrainRow(prompt=prompt, output=f'\\boxed{{{r.answer}}}', answer=str(r.answer), category=r.category, cpcr_stage=CPCR_STAGE[r.category], confidence=r.confidence, source=source, verified=True, train_allowed=True, weight=float(weight), replay_kind=replay_kind, meta=r.meta or {})

def verify_components():
    tests=[
        ('What is 7 * 12?', 'equation_numeric', '84'),
        ('Convert binary 101101 to decimal.', 'numeral_system', '45'),
        ('Convert decimal 45 to binary.', 'numeral_system', '101101'),
        ('Convert 3 km to meter.', 'unit_conversion', '3000'),
        ('Compute bitwise xor 13 and 7.', 'bit_manipulation', '10'),
        ('Compute 6 << 2.', 'bit_manipulation', '24'),
        ('Solve cryptarithm A + A = B. What is B?', 'cryptarithm_deduce', '2'),
        ('On Earth, what is the weight in newtons if mass is 5 kg?', 'physics_gravity', '49'),
    ]
    for p,c,a in tests:
        r=solve_any(p)
        assert r is not None, p
        assert r.category==c, (p,r)
        assert r.answer==a, (p,r)
    print('[OK] CPCR deterministic solvers passed')

verify_components()




In [ ]:
# =============================================================================
# Cell 5 — Synthetic CPCR rows + lossless glyph dual replay rows
# =============================================================================
def dedup_keep_order(xs: Iterable[str]) -> list[str]:
    seen=set(); out=[]
    for x in xs:
        x=' '.join(str(x).strip().split())
        if x and x not in seen:
            seen.add(x); out.append(x)
    return out

def synthetic_prompts(limits=SYNTHETIC_BASE_LIMITS):
    ps=[]
    for a in range(2,80):
        for b in range(2,80):
            ps += [f'What is {a} + {b}?', f'What is {a} * {b}?', f'Calculate {a} + {b}.', f'Compute {a} * {b}.']
            if len([p for p in ps if any(op in p for op in ['+','*'])]) >= limits['equation']*2: break
        if len(ps) >= limits['equation']*2: break
    for n in range(1,1024):
        ps += [f'Convert decimal {n} to binary.', f'Convert binary {n:b} to decimal.']
        if len(ps) > limits['equation']*2 + limits['numeral']*2: break
    for n in range(1,1024):
        ps += [f'Convert {n} km to meter.', f'Convert {n} hour to minute.']
        if len(ps) > limits['equation']*2 + limits['numeral']*2 + limits['unit']*2: break
    bit=[]
    for a in range(1,512):
        for b in range(1,64):
            sh=b%8
            bit += [f'Compute {a} << {sh}.', f'Compute {a} >> {sh}.', f'Compute bitwise xor {a} and {b}.', f'Compute bitwise or {a} with {b}.', f'Compute {a} & {b}.', f'Compute {a} | {b}.', f'What is {a} ^ {b}?']
            if len(bit)>=limits['bit']*2: break
        if len(bit)>=limits['bit']*2: break
    ps += bit
    for m in range(1,512):
        ps += [f'On Earth, what is the weight in newtons if mass is {m} kg?', f'What is the weight in newtons on Earth for mass {m} kg?']
        if len(ps) > limits['equation']*2+limits['numeral']*2+limits['unit']*2+len(bit)+limits['gravity']*2: break
    forms=['Solve cryptarithm A + A = B. What is B?','In the cryptarithm A + A = B, what is B?','For cryptarithm A + A = B, give B.','Cryptarithm: A + A = B. What digit is B?']
    for i in range(limits['cryptarithm']): ps.append(forms[i%len(forms)]+f' Variant {i}.')
    return dedup_keep_order(ps)

def build_verified_rows_from_prompts(prompts, source='synthetic_cpcr'):
    rows=[]
    for p in prompts:
        r=solve_any(p)
        if r and r.verified and r.category in CPCR_STAGE:
            rows.append(make_row(p,r,source=source))
    rows.sort(key=lambda x: (x.cpcr_stage, -x.confidence, x.category, x.prompt))
    return rows

def add_lossless_glyph_replay(rows: list[TrainRow]) -> list[TrainRow]:
    out=[]
    for row in rows:
        out.append(row)
        glyph = text_to_hiero_bytes(row.prompt)
        prompt = (
            'Decode the following lossless Egyptian-hieroglyph byte string as UTF-8, solve the decoded problem, '
            'and return only the final answer in exact boxed form.\n\n'
            f'GlyphBytes: {glyph}'
        )
        out.append(TrainRow(prompt=prompt, output=row.output, answer=row.answer, category=row.category, cpcr_stage=row.cpcr_stage, confidence=row.confidence, source='lossless_hieroglyph_dual_replay', verified=True, train_allowed=True, weight=row.weight*0.85, replay_kind='glyph_dual', meta={'decoded_prompt':row.prompt, 'codec':'egyptian_hieroglyph_byte_utf8'}))
    return out

base_rows = build_verified_rows_from_prompts(synthetic_prompts())
print('[INFO] synthetic verified rows:', len(base_rows), Counter(r.category for r in base_rows))
rows_with_glyph = add_lossless_glyph_replay(base_rows)
print('[INFO] with glyph dual replay:', len(rows_with_glyph), Counter(r.replay_kind for r in rows_with_glyph))




In [ ]:
# =============================================================================
# Cell 6 — Autonomous public/uploaded dataset mining from /kaggle/input
# =============================================================================
def maybe_extract_archives(root: Path, max_archives=80):
    extracted=[]
    if not root.exists(): return extracted
    out_root=BUILD/'extracted_inputs'; out_root.mkdir(parents=True, exist_ok=True)
    archives=list(root.rglob('*.zip'))+list(root.rglob('*.tar.gz'))+list(root.rglob('*.tgz'))
    for i,a in enumerate(archives[:max_archives]):
        dest=out_root/(hashlib.md5(str(a).encode()).hexdigest()[:12])
        if dest.exists():
            extracted.append(dest); continue
        dest.mkdir(parents=True, exist_ok=True)
        try:
            if a.suffix=='.zip':
                with zipfile.ZipFile(a) as z: z.extractall(dest)
            else:
                import tarfile
                with tarfile.open(a) as t: t.extractall(dest)
            extracted.append(dest)
            print('[EXTRACTED]', a, '->', dest)
        except Exception as e:
            print('[WARN] extract failed', a, repr(e))
    return extracted

def iter_candidate_files():
    roots=[INPUT, WORK, BUILD]
    roots += maybe_extract_archives(INPUT)
    exts={'.jsonl','.json','.csv','.txt','.md','.py','.ipynb'}
    seen=set(); n=0
    for root in roots:
        if not root.exists(): continue
        for p in root.rglob('*'):
            if n>=PUBLIC_SCAN_FILE_LIMIT: return
            if not p.is_file() or p.suffix.lower() not in exts: continue
            if p.stat().st_size > 60_000_000: continue
            key=str(p.resolve())
            if key in seen: continue
            seen.add(key); n+=1
            yield p

def extract_text_records_from_file(path: Path, limit_rows=4000):
    rows=[]; suf=path.suffix.lower()
    try:
        if suf=='.jsonl':
            with path.open('r', encoding='utf-8', errors='ignore') as f:
                for i,line in enumerate(f):
                    if i>=limit_rows: break
                    line=line.strip()
                    if not line: continue
                    try: rows.append(json.loads(line))
                    except Exception: pass
        elif suf=='.json':
            obj=json.loads(path.read_text(encoding='utf-8', errors='ignore'))
            if isinstance(obj, list): rows.extend(obj[:limit_rows])
            elif isinstance(obj, dict):
                for k in ['data','rows','train','examples','samples','records']:
                    if isinstance(obj.get(k), list): rows.extend(obj[k][:limit_rows])
                rows.append(obj)
        elif suf=='.csv':
            with path.open('r', encoding='utf-8', errors='ignore', newline='') as f:
                for i,r in enumerate(csv.DictReader(f)):
                    if i>=limit_rows: break
                    rows.append(dict(r))
        elif suf=='.ipynb':
            nb=json.loads(path.read_text(encoding='utf-8', errors='ignore'))
            texts=[]
            for cell in nb.get('cells',[]):
                src=''.join(cell.get('source',[])) if isinstance(cell.get('source'),list) else str(cell.get('source',''))
                if src: texts.append(src)
            rows.append({'text':'\n'.join(texts[:200])})
        else:
            txt=path.read_text(encoding='utf-8', errors='ignore')[:2_000_000]
            rows.append({'text':txt})
    except Exception:
        pass
    return rows

def extract_prompt_answer(obj: Any):
    if not isinstance(obj, dict): return None
    prompt_keys=['prompt','question','problem','input','instruction','query','text']
    answer_keys=['answer','final_answer','target','output','response','label','solution']
    prompt=None; answer=None
    for k in prompt_keys:
        v=obj.get(k)
        if isinstance(v,str) and len(v.strip())>=3:
            prompt=v.strip(); break
    for k in answer_keys:
        v=obj.get(k)
        if isinstance(v,(str,int,float)) and str(v).strip():
            answer=str(v).strip(); break
    # Pull boxed answer if embedded.
    if answer:
        m=re.search(r'\\boxed\{([^{}]+)\}', answer)
        if m: answer=m.group(1).strip()
    if prompt and answer: return prompt, answer
    return None

def mine_public_verified_rows():
    mined=[]; scanned=0
    for f in iter_candidate_files():
        if scanned>=PUBLIC_SCAN_ROW_LIMIT: break
        for obj in extract_text_records_from_file(f):
            scanned+=1
            pa=extract_prompt_answer(obj)
            if not pa: continue
            prompt, ans=pa
            r=solve_any(prompt)
            if not r: continue
            if str(r.answer).strip()==str(ans).strip():
                row=make_row(prompt,r,source='public_exact_verified', replay_kind='public_verified', weight=CATEGORY_WEIGHTS.get(r.category,1)*1.25)
                row.meta={'source_file':str(f)}
                mined.append(row)
    # Dedupe
    seen=set(); out=[]
    for r in mined:
        k=(r.prompt,r.answer,r.category,r.replay_kind)
        if k not in seen:
            seen.add(k); out.append(r)
    print('[INFO] public exact-verified rows:', len(out), Counter(r.category for r in out))
    return out

public_rows = mine_public_verified_rows()
all_verified_rows = rows_with_glyph + add_lossless_glyph_replay(public_rows)
print('[INFO] all verified rows before replay:', len(all_verified_rows), Counter(r.category for r in all_verified_rows))




In [ ]:
# =============================================================================
# Cell 7 — Weighted replay expansion and JSONL audit
# =============================================================================
def row_to_dict(r: TrainRow):
    d=asdict(r); d['meta']=d.get('meta') or {}; return d

def audit_rows(rows: list[TrainRow]):
    errors=[]; cats=Counter(); stages=Counter(); kinds=Counter()
    last_stage=-1
    for i,r in enumerate(rows):
        cats[r.category]+=1; stages[str(r.cpcr_stage)]+=1; kinds[r.replay_kind]+=1
        if r.category not in CPCR_STAGE: errors.append(f'{i}: bad category {r.category}')
        if not r.output.startswith('\\boxed{') or not r.output.endswith('}'): errors.append(f'{i}: bad output {r.output}')
        if not r.verified or not r.train_allowed: errors.append(f'{i}: unverified/train blocked')
        if r.cpcr_stage < last_stage: errors.append(f'{i}: CPCR stage regression')
        last_stage=r.cpcr_stage
    required=set(CPCR_STAGE)
    missing=required-set(cats)
    if missing: errors.append(f'missing categories {sorted(missing)}')
    return {'ok':not errors,'row_count':len(rows),'category_counts':dict(cats),'stage_counts':dict(stages),'replay_kind_counts':dict(kinds),'errors':errors[:50]}

REPLAY_MULT = {
    'numeral_system': 10,
    'unit_conversion': 10,
    'physics_gravity': 8,
    'equation_numeric': 7,
    'cryptarithm_deduce': 10,
    'bit_manipulation': 6,
}

def expand_replay(rows: list[TrainRow], max_rows=MAX_TRAIN_ROWS):
    out=[]
    for r in rows:
        mult=REPLAY_MULT.get(r.category, 2)
        if r.replay_kind=='glyph_dual': mult=max(1, mult//2)
        if r.replay_kind=='public_verified': mult=max(mult, 6)
        for j in range(mult):
            nr=TrainRow(**row_to_dict(r))
            nr.weight=float(r.weight)*(1.0 if j==0 else 0.82)
            out.append(nr)
            if len(out)>=max_rows: break
        if len(out)>=max_rows: break
    random.shuffle(out)
    out.sort(key=lambda x:(x.cpcr_stage, x.category))
    return out

train_rows = expand_replay(all_verified_rows)
report=audit_rows(train_rows)
print(json.dumps(report, indent=2))
assert report['ok'], report

train_jsonl = BUILD/'own_lossless_rank32_train.jsonl'
with train_jsonl.open('w', encoding='utf-8') as f:
    for r in train_rows:
        f.write(json.dumps(row_to_dict(r), ensure_ascii=False)+'\n')
(BUILD/'own_lossless_rank32_audit.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('[OK] wrote', train_jsonl, train_jsonl.stat().st_size)




In [ ]:
# =============================================================================
# Cell 8 — Base model discovery from uploaded Kaggle datasets
# =============================================================================
def config_looks_like_base_model(cfg_path: Path) -> bool:
    try:
        cfg=json.loads(cfg_path.read_text(encoding='utf-8', errors='ignore'))
    except Exception:
        return False
    if 'peft_type' in cfg or 'base_model_name_or_path' in cfg:
        return False
    return bool({'architectures','model_type','hidden_size','num_hidden_layers','vocab_size','auto_map'} & set(cfg.keys()))

def score_model_dir(path: Path) -> int:
    s=str(path).lower(); score=0
    for key,pts in [('nemotron',150),('nano',70),('30b',120),('a3b',90),('reasoning',80),('bf16',60),('omni',35),('nvidia',45),('llama',20)]:
        if key in s: score+=pts
    for bad in ['adapter','lora','checkpoint','trainer','submission','optimizer','corpus','bundle']:
        if bad in s: score-=250
    return score

def discover_base_model():
    candidates=[]
    roots=[INPUT, WORK]
    for root in roots:
        if not root.exists(): continue
        for cfg in root.rglob('config.json'):
            if config_looks_like_base_model(cfg):
                candidates.append((score_model_dir(cfg.parent), cfg.parent))
    candidates=sorted(candidates, key=lambda x:x[0], reverse=True)
    print('[MODEL CANDIDATES]')
    for score,d in candidates[:30]: print(score, d)
    if candidates:
        base=str(candidates[0][1])
        os.environ['TRANSFORMERS_OFFLINE']='1'
        os.environ['HF_HUB_OFFLINE']='1'
        os.environ['HF_DATASETS_OFFLINE']='1'
        print('[INFO] offline local model mode enabled')
        return base
    if ALLOW_HF_DOWNLOAD:
        # last-resort HF id; local mounted model is strongly preferred.
        print('[WARN] no mounted base model found; ALLOW_HF_DOWNLOAD=True, using HF repo id')
        return 'nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16'
    mounted=list(INPUT.iterdir()) if INPUT.exists() else []
    raise SystemExit(
        '[FAIL] No local Nemotron base model config.json found in /kaggle/input. '
        'Add the full Nemotron base model as a Kaggle Input/Model, not only corpus/adapters. Mounted inputs: ' + ', '.join(str(x) for x in mounted[:25])
    )

BASE_MODEL = discover_base_model()
BASE_MODEL_IS_LOCAL = Path(BASE_MODEL).exists()
print('[SELECTED BASE_MODEL]', BASE_MODEL)
print('[BASE_MODEL_IS_LOCAL]', BASE_MODEL_IS_LOCAL)
(BUILD/'base_model.txt').write_text(BASE_MODEL, encoding='utf-8')




In [ ]:
# =============================================================================
# Cell 9 — Dataset, trainer, and answer-only weighted loss
# =============================================================================
import torch
from torch.utils.data import Dataset as TorchDataset

class RowDataset(TorchDataset):
    def __init__(self, rows: list[TrainRow], tokenizer, max_len: int):
        self.rows=rows; self.tok=tokenizer; self.max_len=max_len
    def __len__(self): return len(self.rows)
    def _prompt(self, r: TrainRow):
        return (
            '### Instruction:\n'
            'Solve the problem. Return only the final answer in exact LaTeX boxed form.\n\n'
            f'### CPCR Category:\n{r.category}\n\n'
            f'### Replay Kind:\n{r.replay_kind}\n\n'
            f'### Problem:\n{r.prompt}\n\n'
            '### Answer:\n'
        )
    def __getitem__(self, idx):
        r=self.rows[idx]
        prompt=self._prompt(r); full=prompt+r.output+(self.tok.eos_token or '')
        prompt_ids=self.tok(prompt, add_special_tokens=False, truncation=True, max_length=self.max_len)['input_ids']
        tok=self.tok(full, add_special_tokens=False, truncation=True, max_length=self.max_len, padding='max_length')
        input_ids=tok['input_ids']; labels=list(input_ids)
        for i in range(min(len(prompt_ids), len(labels))): labels[i] = -100
        pad_id=self.tok.pad_token_id
        if pad_id is not None:
            for i,v in enumerate(input_ids):
                if v==pad_id: labels[i] = -100
        return {'input_ids': torch.tensor(input_ids), 'attention_mask': torch.tensor(tok['attention_mask']), 'labels': torch.tensor(labels), 'sample_weight': torch.tensor(float(r.weight), dtype=torch.float32), 'category': r.category}

class WeightedCETrainerMixin:
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        weights=inputs.pop('sample_weight', None)
        inputs.pop('category', None)
        outputs=model(**inputs)
        logits=outputs.logits
        labels=inputs['labels']
        shift_logits=logits[..., :-1, :].contiguous()
        shift_labels=labels[..., 1:].contiguous()
        loss_fct=torch.nn.CrossEntropyLoss(reduction='none')
        token_loss=loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)).view(shift_labels.shape)
        mask=(shift_labels!=-100).float()
        row_loss=(token_loss*mask).sum(dim=1)/mask.sum(dim=1).clamp_min(1.0)
        if weights is not None:
            row_loss=row_loss*weights.to(row_loss.device)
        loss=row_loss.mean()
        return (loss, outputs) if return_outputs else loss

class TimeBudgetCallback:
    def __init__(self, seconds, buffer=480):
        self.start=time.time(); self.seconds=seconds; self.buffer=buffer
    def on_step_end(self, args, state, control, **kwargs):
        if time.time()-self.start > max(1, self.seconds-self.buffer):
            print('[TIME] stopping for final save')
            control.should_training_stop=True
            control.should_save=True
        return control




In [ ]:
# =============================================================================
# Cell 10 — Train own rank-32 adapter with lossless replay + min-logprob replay
# =============================================================================
os.environ.setdefault('TRANSFORMERS_ATTENTION_IMPLEMENTATION','eager')
os.environ.setdefault('TORCH_BACKENDS_CUDA_ENABLE_FLASH_SDP','0')

def load_training_stack():
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
    from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
    print('[INFO] transformers', transformers.__version__)
    return AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training = load_training_stack()

class WeightedTrainer(WeightedCETrainerMixin, Trainer):
    pass

def load_model_and_tokenizer(base_model: str):
    local_only = Path(base_model).exists() or os.environ.get('TRANSFORMERS_OFFLINE') == '1'
    tok=AutoTokenizer.from_pretrained(base_model, trust_remote_code=True, local_files_only=local_only)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    model_kwargs={
        'trust_remote_code':True,
        'torch_dtype':torch.bfloat16,
        'attn_implementation':'eager',
        'local_files_only':local_only,
    }
    # Use bitsandbytes only if importable; if unavailable, normal bf16 load.
    try:
        from transformers import BitsAndBytesConfig
        import bitsandbytes  # noqa
        model_kwargs['quantization_config']=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs['device_map']='auto'
        print('[INFO] using 4-bit NF4 QLoRA')
    except Exception as e:
        print('[WARN] bitsandbytes unavailable; using default model load:', repr(e))
        if torch.cuda.is_available(): model_kwargs['device_map']='auto'
    try:
        model=AutoModelForCausalLM.from_pretrained(base_model, **model_kwargs)
    except ValueError as e:
        msg=str(e)
        if 'dispatched on the CPU or the disk' in msg:
            raise SystemExit('[FAIL] The selected base model does not fit available GPU memory under 4-bit device_map=auto. Use Kaggle T4x2/L4/A100 or a larger GPU. Local 6GB RTX 4050 should run corpus-only smoke, not 30B training.')
        raise
    if hasattr(model.config, 'use_cache'): model.config.use_cache=False
    try:
        model=prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    except Exception as e:
        print('[WARN] prepare_model_for_kbit_training skipped:', repr(e))
    lora=LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none', task_type=TaskType.CAUSAL_LM, target_modules=TARGET_MODULES)
    model=get_peft_model(model, lora)
    model.print_trainable_parameters()
    return model,tok

model, tokenizer = load_model_and_tokenizer(BASE_MODEL)

phase1_rows = train_rows[:]
if SMOKE_TEST_MODE:
    phase1_rows = phase1_rows[:64]

ds1 = RowDataset(phase1_rows, tokenizer, MAX_SEQ_LENGTH if not SMOKE_TEST_MODE else 128)
args1 = TrainingArguments(
    output_dir=str(BUILD/'trainer_phase1'),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM if not SMOKE_TEST_MODE else 1,
    learning_rate=LEARNING_RATE_PHASE1,
    max_steps=MAX_STEPS_SMOKE if SMOKE_TEST_MODE else -1,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to=[],
    remove_unused_columns=False,
    save_safetensors=True,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
)
trainer=WeightedTrainer(model=model, args=args1, train_dataset=ds1, callbacks=[TimeBudgetCallback(TRAINING_TIME_BUDGET_SECONDS, TRAINING_END_BUFFER_SECONDS)])
print('[TRAIN] phase 1 SFT maximize')
trainer.train()
trainer.save_model(str(BUILD/'phase1_adapter'))
tokenizer.save_pretrained(str(BUILD/'phase1_adapter'))

@torch.no_grad()
def score_min_logprob(rows: list[TrainRow], limit=MIN_LOGPROB_EVAL_ROWS):
    model.eval(); scored=[]
    sample=rows[:]
    random.shuffle(sample); sample=sample[:limit]
    tmpds=RowDataset(sample, tokenizer, MAX_SEQ_LENGTH if not SMOKE_TEST_MODE else 128)
    for i in range(len(tmpds)):
        item=tmpds[i]
        batch={k:v.unsqueeze(0).to(model.device) for k,v in item.items() if k in ['input_ids','attention_mask','labels']}
        out=model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        logits=out.logits[:, :-1, :]
        labels=batch['labels'][:, 1:]
        logp=torch.log_softmax(logits, dim=-1)
        mask=labels!=-100
        if mask.sum()==0: continue
        safe_labels=labels.clamp_min(0)
        tok_logp=logp.gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)[mask]
        scored.append((float(tok_logp.min().detach().cpu()), sample[i]))
    scored.sort(key=lambda x:x[0])
    return scored

print('[SCORE] minimum-logprob rows')
scored = score_min_logprob(phase1_rows, limit=512 if SMOKE_TEST_MODE else MIN_LOGPROB_EVAL_ROWS)
minlog_rows=[r for _,r in scored[:(32 if SMOKE_TEST_MODE else MIN_LOGPROB_REPLAY_TOP_N)]]
anchors=[r for r in phase1_rows if r.cpcr_stage==0]
random.shuffle(anchors)
anchor_take=int(len(minlog_rows)*ANCHOR_REPLAY_MIX)
phase2_rows=[]
for r in minlog_rows:
    for _ in range(MIN_LOGPROB_REPLAY_MULTIPLIER):
        nr=TrainRow(**row_to_dict(r)); nr.weight*=1.75; nr.replay_kind='min_logprob_replay'; phase2_rows.append(nr)
phase2_rows += anchors[:anchor_take]
random.shuffle(phase2_rows)
print('[INFO] phase2 rows', len(phase2_rows), Counter(r.category for r in phase2_rows))

(BUILD/'minimum_logprob_replay_rows.jsonl').write_text('\n'.join(json.dumps(row_to_dict(r), ensure_ascii=False) for r in phase2_rows), encoding='utf-8')

if phase2_rows:
    ds2=RowDataset(phase2_rows[:256] if SMOKE_TEST_MODE else phase2_rows, tokenizer, MAX_SEQ_LENGTH if not SMOKE_TEST_MODE else 128)
    args2=TrainingArguments(
        output_dir=str(BUILD/'trainer_phase2_minlogprob'),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM if not SMOKE_TEST_MODE else 1,
        learning_rate=LEARNING_RATE_PHASE2,
        max_steps=MAX_STEPS_SMOKE if SMOKE_TEST_MODE else -1,
        num_train_epochs=1,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        report_to=[],
        remove_unused_columns=False,
        save_safetensors=True,
        bf16=torch.cuda.is_available(),
        gradient_checkpointing=True,
    )
    trainer2=WeightedTrainer(model=model, args=args2, train_dataset=ds2, callbacks=[TimeBudgetCallback(TRAINING_TIME_BUDGET_SECONDS, TRAINING_END_BUFFER_SECONDS)])
    print('[TRAIN] phase 2 minimum-logprob replay')
    trainer2.train()

# Force final save.
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR))
print('[OK] saved own adapter', ADAPTER_DIR)




In [ ]:
# =============================================================================
# Cell 11 — Strict own-adapter validation and packaging
# =============================================================================
def validate_own_adapter(adapter_dir: Path):
    errors=[]
    cfg=adapter_dir/'adapter_config.json'
    st=adapter_dir/'adapter_model.safetensors'
    if not cfg.exists(): errors.append('missing adapter_config.json')
    if not st.exists(): errors.append('missing adapter_model.safetensors')
    if st.exists() and st.stat().st_size < 1024: errors.append('adapter_model.safetensors too small')
    if cfg.exists():
        c=json.loads(cfg.read_text())
        r=int(c.get('r', c.get('rank', -1)))
        if r<1 or r>32: errors.append(f'rank out of contract: {r}')
        if c.get('peft_type') not in [None,'LORA'] and 'LORA' not in str(c.get('peft_type')): errors.append(f'not LoRA: {c.get("peft_type")}')
    try:
        from safetensors.torch import load_file
        tensors=load_file(str(st), device='cpu') if st.exists() else {}
        if not tensors: errors.append('no tensors in safetensors')
        lora_keys=[k for k in tensors if 'lora_' in k.lower()]
        if not lora_keys: errors.append('no LoRA tensors found')
        nonzero=0
        for k,v in tensors.items():
            try:
                if float(v.abs().sum())>0: nonzero+=1
            except Exception: pass
        if nonzero==0: errors.append('all tensors are zero')
        return {'ok':not errors,'errors':errors,'tensor_count':len(tensors),'lora_tensor_count':len(lora_keys),'nonzero_tensor_count':nonzero,'size':st.stat().st_size if st.exists() else 0}
    except Exception as e:
        errors.append(f'safetensors validation failed: {type(e).__name__}: {e}')
        return {'ok':False,'errors':errors}

validation=validate_own_adapter(ADAPTER_DIR)
print(json.dumps(validation, indent=2))
assert validation['ok'], validation

# Flat competition submission: only root-level adapter_config + adapter_model.safetensors.
for p in [SUBMISSION_ZIP, CHECKPOINT_ZIP]:
    if p.exists(): p.unlink()
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ADAPTER_DIR/'adapter_config.json', 'adapter_config.json')
    z.write(ADAPTER_DIR/'adapter_model.safetensors', 'adapter_model.safetensors')
with zipfile.ZipFile(CHECKPOINT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in ADAPTER_DIR.rglob('*'):
        if p.is_file(): z.write(p, str(Path('own_lossless_rank32_adapter')/p.relative_to(ADAPTER_DIR)))

# Extract validation.
with zipfile.ZipFile(SUBMISSION_ZIP) as z:
    names=sorted(z.namelist())
assert 'adapter_config.json' in names and 'adapter_model.safetensors' in names, names

manifest={
    'ok': True,
    'own_adapter_only': OWN_ADAPTER_ONLY,
    'base_model': BASE_MODEL,
    'adapter_dir': str(ADAPTER_DIR),
    'submission_zip': str(SUBMISSION_ZIP),
    'submission_size': SUBMISSION_ZIP.stat().st_size,
    'checkpoint_zip': str(CHECKPOINT_ZIP),
    'checkpoint_size': CHECKPOINT_ZIP.stat().st_size,
    'validation': validation,
    'train_jsonl': str(train_jsonl),
    'train_rows': len(train_rows),
    'smoke_test_mode': SMOKE_TEST_MODE,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
print('[OK] FINAL SUBMISSION:', SUBMISSION_ZIP)
print('[OK] CHECKPOINT:', CHECKPOINT_ZIP)




## Final notes

This notebook creates an **own** rank-32 LoRA adapter. It does not submit a copied public adapter. The “lossless” property is implemented in the data substrate as a deterministic UTF-8 ↔ Egyptian-hieroglyph byte codec. The LoRA learns to retain and use that representation through dual replay, while the deterministic codec itself remains the source of exact reversibility.
